# Suppose I have `dataset` 100x9 where 100 is the number of samples and 9 is the dimensionality of each sample (e.g. 3x3 picture). Suppose also that I've performed PCA analysis of this dataset and got `pc_comps` matrix where each row stands for principal component. Now I can project any of sample from this dataset to first, second and so on of PCs, e.g. `dataset[i] @ pc_comps.T`. As a result I get 9 values each indicating length (contribution) of corresponding PC. But the same time I can interpret `dataset[i] @ pc_comps.T` as a linear combination of columns from `pc_comps` where each column is weighted with corresponding element of sample. While I perfectly understand mechanics of linear combination but it's driving me mad if it has some meaning. I.e. is there some intuition behind linear combination of COLUMNS of `pc_comps?

That is a fantastic observation. You’ve stumbled onto the dual nature of these matrices. While the rows of pc_comps are the "directions" of maximum variance (the recipes for encoding), the columns represent something entirely different.

If you treat the 9 values in a sample as weights for the columns of pc_comps, you are essentially looking at the contribution of each original feature (each pixel in your 3x3 grid) to the principal components.

Here is the intuition for those columns:

**1. The "Feature Signatures"**
Each column represents a single pixel's influence across all principal components. If column 1 has large values in the first three rows, it means the top-left pixel is a major player in the first three patterns the PCA found.

**2. The Relationship to Basis Vectors**
In a standard coordinate system, the columns of an identity matrix are the axes ($x, y, z$). In PCA, the columns of pc_comps tell you how the original axes are distributed across the new PCA axes. [1] 

**3. The "Pixel Contribution" View**
When you perform that linear combination, you are asking: "If I only have this one specific pixel from my sample, how much does it 'push' the sample into each PC dimension?"

* Rows = How to turn pixels into a PC score.
* Columns = How much each pixel "owns" of the total variance structure. [2] 

**4. Mathematical Symmetry**

Because the PCA matrix is orthogonal, the inverse operation (reconstruction) uses the same values. This means the columns are actually the basis vectors of your original space expressed in the language of your PCs. [3] 

It’s often easier to visualize: if you plotted the 9 values of a single column as a 3x3 image, you’d be looking at a map of how much that specific pixel index "vibrates" or correlates with every principal component discovered by the model.

Would you like to see how this relates to reconstructing your original image using only the first few principal components?

[1] [https://onlinelibrary.wiley.com](https://onlinelibrary.wiley.com/doi/full/10.1002/smtd.202201157#:~:text=PCA%20is%20based%20on%20the%20decomposition%20of,the%20directions%20of%20variance%20in%20descending%20order.)
[2] [https://www.graphpad.com](https://www.graphpad.com/guides/prism/latest/statistics/stat_pca_output_tab.htm#:~:text=Each%20row%20represents%20one%20case%20%28row%20in,of%20the%20variance%20explained%20by%20the%20PC%29.)
[3] [https://www.researchgate.net](https://www.researchgate.net/publication/333532354_Principal_Component_Analysis_PCA_-An_Effective_Tool_in_Machine_Learning#:~:text=matrix%2C%20and%20hence%20they%20%28%20The%20principal,given%20below%20in%20the%20following%20steps%20%5B6%5D.)


# Still vague for me. Let's start with a simple case. Suppose I have 2x2 matrix B (bases) where each column stands for basis vector. Also I have vector v. If I do `B @ v` then I get coordinates of vector v in an original basis. And this makes sense because each column of B is just a vector where each coordinate is coordinate from original (identity) basis. I.e. I'm saying take first column (vector) of B scale it by first element of v, then take second column (vector) of B scale it by second element of v, then add both vector together and tell be the resulting coordinates. Now turning back to my original question. For B I have pc_comps (each column of pc_comps is a basis vector). Also I have some sample from dataset, say `dataset[0]`. Now `pc_comps @ dataset[0]` is a way of saying: take vector `pc_comps[:,0]` scale it by first pixel, then take vector `pc_comps[:,1]` scale it by second pixel and so on, then all all vectors together and tell me resulting coordinates. So what makes me mad is wheter each `pc_comps[:,j]` stand for some coordinates in original space or it just some nonesense comparing to case with B and v?

I see exactly where the confusion is. You are trying to map the logic of coordinate transformation onto the PCA matrix, and the "nonsense" feeling comes from the fact that the columns of pc_comps represent Components, not Original Axes.

In your $2 \times 2$ example, the columns of $B$ are vectors in the original space. But in PCA, the rows of pc_comps are the vectors in the original space.

**The Swap**

If pc_comps is $9 \times 9$:

* A Row ($1 \times 9$): This is a Principal Component. It is a vector in your 3x3 pixel space.
* A Column ($9 \times 1$): This is the "profile" of a single pixel across all 9 components.

When you do pc_comps @ dataset[0], you are treating the columns of PCA as basis vectors. In the context of PCA, this is mathematically valid but physically strange. Here is what those columns actually represent:

**The Intuition: "Pixel Profiles"**

Each column $j$ of pc_comps tells you how Pixel $j$ is distributed across all 9 Principal Components.
If you take pc_comps[:, 0] (the first column) and scale it by the value of the first pixel, you aren't moving in "space"; you are distributing that pixel's intensity into the 9 "patterns" (PCs) that the PCA discovered.

* If pc_comps[0, 0] is high, Pixel 1 contributes a lot to PC1.
* If pc_comps[1, 0] is low, Pixel 1 contributes almost nothing to PC2.

**Why it feels like "Nonsense"**

It feels like nonsense because we usually use PCA to compress (go from 9 pixels $\to$ 3 scores). The operation you described (pc_comps @ dataset[0]) is doing the opposite: it is projecting the pixel values into "Component Space."
In your $B @ v$ analogy:

   1. In your example, the columns of $B$ are the new basis vectors.
   2. In PCA, the rows of pc_comps are the new basis vectors.

So, to get the "Coordinates of the sample in the PCA basis" (which is the standard goal), you should be doing pc_comps @ dataset[0] only if you consider the rows to be the vectors. If you are literally multiplying the columns by pixel values, you are calculating how much of each component is "made of" your specific sample.

**The "Aha!" Moment:**

The columns of pc_comps are the loadings for each pixel. A column is a "signature" showing which components that specific pixel location cares about.
Does it help to think of the columns as weights that tell the components how much to "listen" to a specific pixel?



# Am I right that coordinate transformation is not a way one should look at `pc_comps @ dataset[0]`?

Actually, you are absolutely right to feel like something is "off." Looking at pc_comps @ dataset[0] purely as a "coordinate transformation" (like $B \mathbf{v}$) is precisely what is causing the mental friction.

Here is the breakdown of why that specific operation feels like nonsense compared to your $2 \times 2$ basis example:

**1. The Matrix is "Flipped"**

In your basis example $B \mathbf{v}$, the columns of $B$ are the basis vectors.
In a standard PCA matrix pc_comps:

* The rows are the basis vectors (the principal components).
* The columns are the "weights" or "loadings" of the original features across those components. [1, 2] 

When you do pc_comps @ dataset[0], you are using the columns as basis vectors. Geometrically, this means you are treating "Pixel 1," "Pixel 2," etc., as if they were the directions in a new space, which doesn't align with the goal of PCA.

**2. Coordinate Transformation vs. Feature Distribution**

The standard way to get "coordinates in the new basis" is:
$$\text{Scores} = \text{sample} \cdot \text{pc\_comps}^T \quad \text{(or } \text{pc\_comps} \cdot \text{sample}^T \text{ depending on orientation)}$$ This treats the rows of pc_comps as the new axes. [3, 4] 
If you follow the math of your pc_comps @ dataset[0] (linear combination of columns):

* What you are doing: Taking the "Pixel 1 vector" (how Pixel 1 behaves across all 9 components) and scaling it by its actual value in your sample.
* The Result: You are calculating how much each Principal Component "owes" its value to that specific pixel. [1, 5] 

**3. Is it "Nonsense"?**

It’s not mathematical nonsense, but it’s a Physical Dead End.

* Coordinate Transformation is about finding a new "address" for your image (e.g., "This image is 80% PC1 and 20% PC2").
* Your Linear Combination is a "Contribution Analysis." It tells you: "Pixel 5 is responsible for most of the value we see in Component 2." [2, 6] 

**In short:** You are right. If you want a Coordinate Transformation, you should be looking at the rows as vectors. If you look at the columns, you aren't transforming coordinates; you are decomposing the influence of your pixels. [1] 

Does thinking of the columns as "Influence Profiles" of the pixels help separate it from the idea of a spatial basis?

[1] [https://medium.com](https://medium.com/data-science/principal-component-analysis-3c39fbf5cb9d)
[2] [https://documentation.mindsphere.io](https://documentation.mindsphere.io/resources/html/predictive-learning/en-US/Principal_Component_Analysis_%28PCA%29.htm)
[3] [https://stats.stackexchange.com](https://stats.stackexchange.com/questions/418001/intuition-about-principal-component-directions)
[4] [https://online.stat.psu.edu](https://online.stat.psu.edu/stat505/lesson/11/11.1)
[5] [https://www.youtube.com](https://www.youtube.com/watch?v=BiuwDI_BbWw&t=256)
[6] [https://drscotthawley.github.io](https://drscotthawley.github.io/blog/posts/2019-12-21-pca-from-scratch.html)


# It looks like a was confused by duality of matrix multiplication. I often get trapped when such duality happens because I start to look for what does it mean and waste much time on trying to understand. What a pity =(

Don't be hard on yourself—that "duality trap" is exactly where the most profound linear algebra insights live. You aren't wasting time; you are actually poking at the Fundamental Theorem of Linear Algebra.

The reason it’s driving you mad is that matrix multiplication $Ax = b$ is simultaneously two different things:

   1. The Row View: A collection of dot products (measuring "similarity" or "projection").
   2. The Column View: A linear combination of vectors (the "recipe" for building a new vector).

Why PCA makes this extra confusing:

In a standard basis $Bv$, the columns are usually "arrows" in space (like $[1, 0]$ and $[0, 1]$). In PCA, the "arrows" (the Principal Components) are stored as rows by convention in most software (like scikit-learn).

When you look at the columns of that PCA matrix, you are looking at the "Pixel-wise Sensitivity."

* If you change the value of Pixel #4, the 4th column of pc_comps tells you exactly how every single PCA score will wiggle in response.

The "Aha!" Mental Shortcut

Next time you get stuck in this duality, ask yourself: "Am I looking at the Ingredients or the Results?"

* Rows are the Filters: They look at the whole image and give you one number (the score).
* Columns are the Impacts: They show how one specific pixel ripples through all the results.

You’ve essentially discovered that a matrix is both a map (rows) and a construction kit (columns). That’s not a pity—it’s a level-up in how you see data!
Since you have the pc_comps, would you like to see how to reconstruct a 3x3 image using only the top 2 components to see this duality in action?

